In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# A Parser for Propositional Logic

## Representing Formulas

In Python, formulas are represented as nested tuples.  In F#, we use a *discriminated union*.  A discriminated union is a type whose values are built from a fixed set of *constructors*.  Every constructor can carry arguments.  The type `Formula` given below has one constructor for every kind of formula:

| Formula             | F# representation |
|:--------------------|:------------------|
| $p$                 | `Var "p"`         |
| $\top$              | `Verum`           |
| $\bot$              | `Falsum`          |
| $\neg f$            | `Not f`           |
| $f \wedge g$        | `And (f, g)`      |
| $f \vee g$          | `Or (f, g)`       |
| $f \rightarrow g$   | `Imp (f, g)`      |
| $f \leftrightarrow g$ | `Iff (f, g)`    |
| $f \oplus g$        | `Xor (f, g)`      |

The type is *recursive*: the arguments of the constructors `Not`, `And`, `Or`, `Imp`, `Iff`, and `Xor` are again formulas.  For example, the formula $(p \rightarrow q) \wedge \neg r$ is represented as
```
And (Imp (Var "p", Var "q"), Not (Var "r"))
```
In contrast to nested tuples, the F# compiler checks that every value of type `Formula` is a well-formed formula.  For example, it is impossible to apply `Not` to two arguments.

In [2]:
type Formula =
    | Var    of string             // propositional variable
    | Verum                        // ⊤
    | Falsum                       // ⊥
    | Not    of Formula            // ¬
    | And    of Formula * Formula  // ∧
    | Or     of Formula * Formula  // ∨
    | Imp    of Formula * Formula  // →
    | Iff    of Formula * Formula  // ↔
    | Xor    of Formula * Formula  // ⊕

In [3]:
And (Imp (Var "p", Var "q"), Not (Var "r"))

And (Imp (Var "p", Var "q"), Not (Var "r"))

## The Tokenizer

The function `tokenize` takes one argument:
- `s` is a string that is supposed to represent a formula from propositional logic.

The function returns the list of *tokens* of `s`.  A token is either
- an *identifier*, i.e. a string that starts with a letter that is followed by letters, digits, or one of the characters `<`, `>`, and `,`,
- one of the logical operators `⊤`, `⊥`, `∧`, `∨`, `¬`, `→`, `↔`, `⊕`, or
- a parenthesis.

Blanks and tabs are skipped.

**Implementation:** The tokens are specified by a *regular expression* that has three groups:
1. The first group matches blanks and tabs.
2. The second group matches identifiers.
3. The third group matches the logical operators and parentheses.

The option `RegexOptions.IgnorePatternWhitespace` allows us to write the regular expression on several lines and to add comments.  Therefore, the blank that is to be matched in the first group has to be escaped as `\ `.

`Regex.Matches` returns all matches of the regular expression in `s`.  For every match `m`, `m.Groups[k].Success` is true if the $k$-th group has matched.  If the first group has matched, the match is whitespace and is discarded.  Otherwise, the matched string `m.Value` is a token.

In [4]:
open System.Text.RegularExpressions

let tokenize (s: string) : string list =
    let lexSpec = """([\ \t]+)                  |  # blanks and tabs
                     ([A-Za-z][A-Za-z0-9<>,]*)  |  # identifier
                     ([⊤⊥∧∨¬→↔⊕()])                # junctors, '(', ')'
                  """
    [ for m in Regex.Matches(s, lexSpec, RegexOptions.IgnorePatternWhitespace) do
        if not m.Groups[1].Success then
            m.Value ]

In [5]:
tokenize "p ↔ q ↔ r"

["p"; "↔"; "q"; "↔"; "r"]

In [6]:
tokenize "¬(a<1,2> ∧ b) → ⊥"

["¬"; "("; "a<1,2>"; "∧"; "b"; ")"; "→"; "⊥"]

The function `isPropVar` takes one argument:
- `s` is a string.

The function returns `true` if `s` can be interpreted as a propositional variable, i.e. if `s` is an identifier.
`Regex.IsMatch` checks whether the regular expression matches some part of `s`.  The anchors `^` and `$` ensure that the complete string `s` is matched.

In [7]:
let isPropVar (s: string) : bool =
    Regex.IsMatch(s, "^[A-Za-z][A-Za-z0-9<>,]*$")

In [8]:
isPropVar "a<1,2>", isPropVar "∧"

(true, false)

## The Shunting Yard Algorithm

The parser implements the *shunting yard algorithm* that was invented by Edsger W. Dijkstra.  It uses two stacks:
- `arguments` is a stack of formulas that have already been parsed,
- `operators` is a stack of operators (and opening parentheses) that are still waiting for their arguments.

Before we can define the parser, we need to define the *precedences* of the operators.  The function `precedence` takes one argument:
- `op` is a string representing a logical operator.

It returns the precedence of `op`.  Operators with a higher precedence bind stronger.  For example, as $\wedge$ has a higher precedence than $\vee$, the formula $p \vee q \wedge r$ is parsed as $p \vee (q \wedge r)$.

In [9]:
let precedence (op: string) : int =
    match op with
    | "↔"       -> 1
    | "→"       -> 2
    | "⊕"       -> 3
    | "∨"       -> 4
    | "∧"       -> 5
    | "¬"       -> 6
    | "⊤" | "⊥" -> 7
    | _         -> failwithf "unknown operator %s" op

The function `evalBefore` takes two arguments:
- `stackOp` is the operator on top of the operator stack,
- `nextOp` is the next operator from the input.

The function returns `true` if the operator `stackOp` has to be evaluated before the operator `nextOp` is processed.  This is the case if
- `stackOp` has a higher precedence than `nextOp`, or
- both operators have the same precedence and they are different, or
- both operators are identical and are *left associative*.  The operators `∧`, `∨`, and `⊕` are left associative, while `→` and `↔` are *right associative*.  For example, $p \rightarrow q \rightarrow r$ is parsed as $p \rightarrow (q \rightarrow r)$.

An opening parenthesis on the operator stack is never evaluated: it has to wait for the matching closing parenthesis.

In [10]:
let evalBefore (stackOp: string) (nextOp: string) : bool =
    if stackOp = "(" then
        false
    elif precedence stackOp > precedence nextOp then
        true
    elif precedence stackOp = precedence nextOp then
        if stackOp = nextOp then
            List.contains stackOp ["∧"; "∨"; "⊕"]  // left associative
        else
            true
    else
        false

The function `apply` takes two arguments:
- `op` is a string representing a logical operator,
- `arguments` is the stack of formulas, represented as a list where the top of the stack is the head of the list.

The function takes the arguments of `op` from the stack `arguments`, builds the formula that results from applying `op` to these arguments, and pushes this formula onto the stack.  The new stack is returned.
- The operators `⊤` and `⊥` take no arguments.
- The operator `¬` takes one argument.
- All other operators take two arguments.  As the left argument was pushed first, the right argument `rhs` is on top of the stack.

If there are not enough arguments on the stack, the input is not a well-formed formula and an exception is raised.

In [11]:
let apply (op: string) (arguments: Formula list) : Formula list =
    match op, arguments with
    | "⊤", _               -> Verum :: arguments
    | "⊥", _               -> Falsum :: arguments
    | "¬", f :: rest       -> Not f :: rest
    | "∧", rhs :: lhs :: rest -> And (lhs, rhs) :: rest
    | "∨", rhs :: lhs :: rest -> Or  (lhs, rhs) :: rest
    | "→", rhs :: lhs :: rest -> Imp (lhs, rhs) :: rest
    | "↔", rhs :: lhs :: rest -> Iff (lhs, rhs) :: rest
    | "⊕", rhs :: lhs :: rest -> Xor (lhs, rhs) :: rest
    | _                    -> failwithf "syntax error: missing argument for %s" op

The function `parse` takes one argument:
- `s` is a string that represents a formula from propositional logic.

The function returns the formula represented by `s`, i.e. the *syntax tree* of `s`.  If `s` can not be parsed, an exception is raised.

**Implementation:** The tokens of `s` are processed from left to right.  The function uses three mutable variables, which are all lists that are used as stacks:
- `tokens` contains the tokens that still have to be processed,
- `operators` is the operator stack,
- `arguments` is the argument stack.

For every token `nextOp` taken from `tokens` we distinguish the following cases:
1. If `nextOp` is a propositional variable, it is pushed onto the argument stack.
2. If `nextOp` is `⊤` or `⊥`, or if it is an opening parenthesis, or if the operator stack is empty, then `nextOp` is pushed onto the operator stack.
3. Otherwise, let `stackOp` be the operator on top of the operator stack.
   - If `stackOp` is an opening parenthesis and `nextOp` is a closing parenthesis, the opening parenthesis is removed from the operator stack.
   - If `nextOp` is a closing parenthesis, or if `stackOp` has to be evaluated before `nextOp`, then `stackOp` is removed from the operator stack and applied to its arguments.  The token `nextOp` is put back into the list `tokens`, so that it will be processed again.
   - Otherwise, `nextOp` is pushed onto the operator stack.

Once all tokens have been processed, the remaining operators on the operator stack are applied.  If the input was a well-formed formula, the argument stack now contains exactly one formula, which is the result.

In [12]:
let parse (s: string) : Formula =
    let mutable tokens    = tokenize s
    let mutable operators : string list  = []
    let mutable arguments : Formula list = []
    while not tokens.IsEmpty do
        let nextOp = tokens.Head
        tokens <- tokens.Tail
        if isPropVar nextOp then
            arguments <- Var nextOp :: arguments
        elif nextOp = "⊤" || nextOp = "⊥" || nextOp = "(" || operators.IsEmpty then
            operators <- nextOp :: operators
        else
            let stackOp = operators.Head
            if stackOp = "(" && nextOp = ")" then
                operators <- operators.Tail
            elif nextOp = ")" || evalBefore stackOp nextOp then
                operators <- operators.Tail
                arguments <- apply stackOp arguments
                tokens    <- nextOp :: tokens
            else
                operators <- nextOp :: operators
    for op in operators do
        arguments <- apply op arguments
    match arguments with
    | [ f ] -> f
    | _     -> failwithf "could not parse %s" s

## Testing
The function `testParser` takes one argument:
- `s` is a string representing a formula.

The function parses `s` and prints both `s` and the resulting formula.

In [13]:
let testParser (s: string) : unit =
    printfn "parsing %s" s
    print (parse s)
    printfn ""

In [14]:
testParser "¬⊥"
testParser "¬p ↔ (p → ⊥)"
testParser "¬⊥ ↔ ⊤"
testParser "p ∧ q"
testParser "p ∨ q ∧ r"
testParser "p ∧ q ∨ r"
testParser "p ∧ q → r ∨ s"
testParser "p → q → r"
testParser "p ∧ q ↔ q ∨ p"
testParser "¬(p ∨ q) ↔ ¬p ∨ ¬q"
testParser "¬(p ⊕ q) ↔ (p ↔ q)"
testParser "a<1,2> ↔ b<2,1>"

parsing 

¬⊥

Not Falsum

parsing 

¬p ↔ (p → ⊥)

...

¬(p ⊕ q) ↔ (p ↔ q)

Iff (Not (Xor (Var "p", Var "q")), Iff (Var "p", Var "q"))

parsing 

a<1,2> ↔ b<2,1>

Iff (Var "a<1,2>", Var "b<2,1>")

Strings that are not well-formed formulas are rejected.

In [15]:
parse "p ∧ ∨ q"

Error: System.Exception: syntax error: missing argument for ∧
   at Microsoft.FSharp.Core.PrintfModule.PrintFormatToStringThenFail@1448.Invoke(String message)
   at FSI_0016.parse(String s)
   at <StartupCode$FSI_0019>.$FSI_0019.main@()
   at System.Reflection.MethodBaseInvoker.InterpretedInvoke_Method(Object obj, IntPtr* args)
   at System.Reflection.MethodBaseInvoker.InvokeWithNoArgs(Object obj, BindingFlags invokeAttr)